# Task 07 — Kaggle DCGRU workflow

This notebook separates setup, integrity verification, GPU probing, full training, finalization, and artifact packaging. Only setup and integrity verification are approved initially. Probe, training, and finalization commands never run automatically.

## Stage 1 — Environment and uploaded-dataset setup

In [1]:

from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import torch

# Stage 1 — Environment and uploaded-dataset setup

# Actual Kaggle dataset location
uploaded_project = Path(
    '/kaggle/input/datasets/pirabakaranyathavan/'
    'urbanflow-ai-task07/Capstone-Project_268513X'
)

# Writable project directory
project = Path('/kaggle/working/Capstone-Project_268513X')

# Verify dataset exists
assert uploaded_project.is_dir(), (
    f'Kaggle dataset directory not found: {uploaded_project}'
)

# Verify package manifest exists
assert (uploaded_project / 'kaggle_package_manifest.json').is_file(), (
    'Package manifest not found'
)

# Prevent accidental overwriting
if project.exists():
    raise FileExistsError(
        f'Working directory already exists: {project}. '
        'Inspect it before reusing or removing it.'
    )

# Copy dataset into writable workspace
shutil.copytree(uploaded_project, project)

print('Project copied successfully!')
print('Source:', uploaded_project)
print('Working directory:', project)

# Environment information
print('\n--- Environment ---')
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(
        'GPU memory GiB:',
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        )
    )
else:
    print('WARNING: CUDA is unavailable. Enable GPU in Kaggle settings.')

Project copied successfully!
Source: /kaggle/input/datasets/pirabakaranyathavan/urbanflow-ai-task07/Capstone-Project_268513X
Working directory: /kaggle/working/Capstone-Project_268513X

--- Environment ---
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory GiB: 14.56


## Stage 2 — Package, hash, data, graph, and alignment verification

In [2]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest().upper()

manifest = json.loads((project / 'kaggle_package_manifest.json').read_text())
assert manifest['package_root'] == 'Capstone-Project_268513X'
for entry in manifest['files']:
    path = project / entry['path']
    assert path.is_file(), entry['path']
    assert path.stat().st_size == entry['bytes'], entry['path']
    assert sha256(path) == entry['sha256'], entry['path']
print('Verified package files:', manifest['file_count'])
subprocess.run([sys.executable, 'scripts/run_dcrnn.py', '--integrity-only'], cwd=project, check=True)

Verified package files: 21
{
  "mode": "integrity-only",
  "data_sha256": "8D67A35472DB1719D7D4BE851F2BF64CB21D9C52577C8A6B4B873D43205AF381",
  "split_rows": {
    "train": [
      0,
      23990
    ],
    "validation": [
      23990,
      27417
    ],
    "test": [
      27417,
      34272
    ]
  },
  "graph_manifest": "results/task07_dcrnn/graph_manifest.json",
  "locked_artifact_hashes": {
    "task04_baselines": "48FD54D29C5BDB130F978F2526865649BF247CE8D6C371C8807EE8DC4D5B4321",
    "task05_xgboost": "19AF5FC9C78053D8B7F61DDB7B5D9A7122765328B1403004E3E1E5429037D236",
    "task06_gru": "27C165BDE7FD72E1CD12D21F4D0D1087F9FA1CBCB08E90EE800461C39EC5FCAF"
  },
  "test_inputs_passed_to_task07_model": false,
  "test_predictions_or_metrics_calculated": false
}


CompletedProcess(args=['/usr/bin/python3', 'scripts/run_dcrnn.py', '--integrity-only'], returncode=0)

## Stage 3 — GPU probe (run only after approval)

Supply actual owner session/quota values. Run batch sizes 16, 32, and 64 as separate commands. Probe mode loads only train and validation datasets.

In [6]:

# Stage 3 — GPU feasibility probe (selected batch size 64)

import subprocess
import sys
import time
import torch

# Confirmed Kaggle limits from account screenshots
session_limit_hours = 12.0
remaining_gpu_quota_hours = 24 + 22 / 60

# Probe configuration
batch_size = 64
safety_margin = 0.30

assert torch.cuda.is_available(), "CUDA GPU is not available"
assert project.is_dir(), f"Project directory not found: {project}"

print("=" * 60)
print("UrbanFlow AI — Task 07 GPU Feasibility Probe")
print("=" * 60)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB")
print(f"Session limit: {session_limit_hours:.2f} hours")
print(f"Remaining GPU quota (earlier snapshot): {remaining_gpu_quota_hours:.2f} hours")
print(f"Batch size: {batch_size}")
print(f"Safety margin: {safety_margin:.0%}")
print()

probe_command = [
    sys.executable,
    "scripts/run_dcrnn.py",
    "--probe-only",
    "--device", "cuda",
    "--batch-size", str(batch_size),
    "--num-workers", "2",
    "--warmup-batches", "5",
    "--probe-train-batches", "50",
    "--probe-validation-batches", "20",
    "--session-limit-hours", str(session_limit_hours),
    "--remaining-gpu-quota-hours", str(remaining_gpu_quota_hours),
    "--safety-margin", str(safety_margin),
]

print("Starting GPU probe...")
start_time = time.perf_counter()

try:
    subprocess.run(
        probe_command,
        cwd=project,
        check=True,
    )

    elapsed = time.perf_counter() - start_time

    print("\n" + "=" * 60)
    print("GPU PROBE COMPLETED SUCCESSFULLY")
    print(f"Elapsed time: {elapsed / 60:.2f} minutes")
    print("=" * 60)

except subprocess.CalledProcessError as error:
    print(f"\nGPU probe failed with exit code: {error.returncode}")
    raise

UrbanFlow AI — Task 07 GPU Feasibility Probe
GPU: Tesla T4
GPU memory: 14.56 GiB
Session limit: 12.00 hours
Remaining GPU quota (earlier snapshot): 24.37 hours
Batch size: 64
Safety margin: 30%

Starting GPU probe...
{
  "measured_training_batches": 50,
  "training_batches_per_second": 12.026566766806674,
  "training_batch_seconds": 0.08314924943999813,
  "windows_per_second": 769.7002730756271,
  "validation_windows_per_second": 1685.9156591950255,
  "peak_allocated_bytes": 826457088,
  "peak_reserved_bytes": 983564288,
  "estimated_epoch_seconds": 31.1809685399993,
  "estimated_validation_seconds": 2.0499246099001995,
  "estimated_30_epoch_hours": 0.2770171967741633,
  "estimated_30_epoch_includes": "training, validation, and one checkpoint save per epoch; locked-test finalization is separately unmeasured",
  "checkpoint_save_seconds": 0.011170463000098607,
  "checkpoint_load_resume_seconds": 0.008055691999970804,
  "checkpoint_cpu_rng_restored": true,
  "checkpoint_dataloader_rng_re

## Stage 4 — Full validation-only training (separate owner approval required)

In [7]:

# Stage 4 — Full validation-only DCGRU training

import subprocess
import sys
import time
import torch

approved_batch_size = 64

assert torch.cuda.is_available(), "CUDA GPU is unavailable"
assert (project / "scripts" / "run_dcrnn.py").is_file()

training_command = [
    sys.executable,
    "scripts/run_dcrnn.py",
    "--full",
    "--device", "cuda",
    "--batch-size", str(approved_batch_size),
    "--num-workers", "2",
]

print("=" * 60)
print("UrbanFlow AI — Task 07 Full Training")
print("=" * 60)
print("Model: DCGRU-64, diffusion order 2")
print(f"Batch size: {approved_batch_size}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Maximum epochs: 30")
print("Early-stopping patience: 5")
print("Test evaluation: NOT PERMITTED in this stage")
print("=" * 60)

start_time = time.perf_counter()

subprocess.run(
    training_command,
    cwd=project,
    check=True,
)

elapsed = time.perf_counter() - start_time

print("\nTraining process completed.")
print(f"Elapsed time: {elapsed / 60:.2f} minutes")
print("Review validation results before Stage 5.")

UrbanFlow AI — Task 07 Full Training
Model: DCGRU-64, diffusion order 2
Batch size: 64
GPU: Tesla T4
Maximum epochs: 30
Early-stopping patience: 5
Test evaluation: NOT PERMITTED in this stage
{"epoch": 0, "train_mae_mph": 3.9095205475410424, "train_eligible_count": 55289280, "validation_horizon_mae_mph": {"15": 2.894086601028959, "30": 3.389187799461568, "60": 4.226215392132623}, "selection_score_mph": 3.5031632642077164}
{"epoch": 1, "train_mae_mph": 3.5609267870281545, "train_eligible_count": 55289280, "validation_horizon_mae_mph": {"15": 2.824952249765109, "30": 3.2845761530814466, "60": 4.107942208682503}, "selection_score_mph": 3.405823537176353}
{"epoch": 2, "train_mae_mph": 3.4837058693837215, "train_eligible_count": 55289280, "validation_horizon_mae_mph": {"15": 2.8034575308802747, "30": 3.2218718841719904, "60": 4.010094117706586}, "selection_score_mph": 3.345141177586284}
{"epoch": 3, "train_mae_mph": 3.431024206625136, "train_eligible_count": 55289280, "validation_horizon_ma

## Stage 5 — Locked-test finalization (separate approval required)

In [8]:

# Stage 5 — Locked-test finalization
# Run only after explicitly approving final test evaluation.

import subprocess
import sys
import time

approved_batch_size = 64

best_checkpoint = (
    project
    / "artifacts/task07_dcrnn/checkpoints/dcgru_h64_k2_best.pt"
)

assert best_checkpoint.is_file(), (
    f"Best checkpoint not found: {best_checkpoint}"
)

finalization_command = [
    sys.executable,
    "scripts/run_dcrnn.py",
    "--finalize-only",
    "--device", "cuda",
    "--batch-size", str(approved_batch_size),
    "--num-workers", "2",
]

print("Task 07 — Locked-test finalization")
print("Best checkpoint:", best_checkpoint)
print("Batch size:", approved_batch_size)

start = time.perf_counter()

subprocess.run(
    finalization_command,
    cwd=project,
    check=True,
)

print(
    f"Finalization completed in "
    f"{(time.perf_counter() - start):.2f} seconds"
)

Task 07 — Locked-test finalization
Best checkpoint: /kaggle/working/Capstone-Project_268513X/artifacts/task07_dcrnn/checkpoints/dcgru_h64_k2_best.pt
Batch size: 64
{
  "task": "Task 07 - DCRNN-style direct-head DCGRU",
  "created_utc": "2026-11-12T04:10:47.913492+00:00",
  "configuration": {
    "candidate_name": "dcgru_h64_k2",
    "model": {
      "history_features": 3,
      "hidden_size": 64,
      "sensor_count": 207,
      "sensor_embedding_dim": 8,
      "output_length": 12,
      "future_time_features": 2,
      "diffusion_order": 2,
      "graph_mode": "graph"
    },
    "learning_rate": 0.001,
    "weight_decay": 0.0,
    "batch_size": 64,
    "max_epochs": 30,
    "patience": 5,
    "gradient_clip_norm": 1.0,
    "seed": 268513,
    "deterministic_algorithms": false
  },
  "metadata": {
    "data_sha256": "8D67A35472DB1719D7D4BE851F2BF64CB21D9C52577C8A6B4B873D43205AF381",
    "adjacency_sha256": "A35687C6E15AA228DC45027B0ED2A0EA0F4EC78F573DEB992C595092D12F61B3",
    "sensor_

## Stage 6 — Artifact packaging

After an approved stage completes, archive only its Task 07 results/checkpoints and download the archive before the session ends.

In [9]:

# Stage 6 — Package and verify Task 07 outputs

from pathlib import Path
import shutil
import hashlib

packages = {
    "artifacts": "artifacts/task07_dcrnn",
    "results": "results/task07_dcrnn",
}

for name, relative_dir in packages.items():
    source = project / relative_dir
    assert source.is_dir(), f"Missing directory: {source}"

    archive = Path(
        shutil.make_archive(
            f"/kaggle/working/task07_dcrnn_{name}",
            "zip",
            root_dir=project,
            base_dir=relative_dir,
        )
    )

    digest = hashlib.sha256(archive.read_bytes()).hexdigest()

    print(f"\n{name.upper()}")
    print("Archive:", archive)
    print("Size (MiB):", round(archive.stat().st_size / 1024**2, 2))
    print("SHA256:", digest)



ARTIFACTS
Archive: /kaggle/working/task07_dcrnn_artifacts.zip
Size (MiB): 17.37
SHA256: 97c0bb6ddc4c83b60104b9f226c65a6964f0b65eb269e4a37ace388af776dd24

RESULTS
Archive: /kaggle/working/task07_dcrnn_results.zip
Size (MiB): 0.01
SHA256: a22e9a9e7860ae8790b6a3655182552210007c96456127caa64779b346b7d1ad
